# 01. Preprocessing: Column Selection, Time Split and Leakage Fix

CRISP-DM phase: Data preparation

## 1. Setup and Data Loading

### 1.1 Importing Libraries

In [1]:
#necessary imports
import os
import pandas as pd

### 1.2 Loading the Dataset

`model_table.csv` is the modelling table of the project. The original table had 14,664 rows and 70 columns. It was reduced on purpose to 30 columns (the 4 keys, the target, `flood_this_month` and the 24 features chosen in the EDA) and to the months July 2002 to December 2024, so that the file is easier to open and navigate. This notebook only reads the file and never writes to it. The table is team-only because its flood labels contain EM-DAT data, so it is not stored in this repository. The path is relative to the `notebooks` folder. `pcode` is read as text so the county codes are kept exactly as they are.

In [2]:
#loading the modelling table (pcode as text)
df = pd.read_csv('../data/model_table.csv', dtype={'pcode': str})

In [3]:
#shape of the dataset
df.shape

(12690, 30)

In [4]:
#data type of pcode and number of counties
print(df['pcode'].dtype)
print(df['pcode'].nunique())

str
47


#### Observation

The file has 12,690 rows and 30 columns, and `pcode` is read as text (type `str`) with 47 different counties. This is the reduced file described above, so the next sections check that it has the columns and the months we expect.

## 2. Checking the Columns

### 2.1 Keys, Target and Features

The file has 30 columns: 4 keys, the target, `flood_this_month` and the 24 features chosen in the EDA (notebook 04 of the capstone project). The next cells name them. In the model notebooks `flood_this_month` is used twice: as an extra feature, and as the persistence baseline (predict a flood next month if the county floods this month).

In [5]:
#the four key columns that identify each county-month
keys = ['pcode', 'county', 'year', 'month']

In [6]:
#the target: will the county flood next month?
target = 'flood_next_month'

In [7]:
#flood_this_month: did the county flood this month? (a feature in the model notebooks and the persistence baseline)
baseline = 'flood_this_month'

In [8]:
#the 24 selected features
features = ['rain_mm', 'rain_lag1', 'rain_lag2', 'rain_pct_normal', 'neighbour_rain',
            'ndvi_pct_normal', 'oni', 'dmi', 'month_sin', 'month_cos',
            'soil_moisture_l1', 'runoff_rank', 'discharge_rank',
            'pct_flat', 'area_km2', 'lc_tree', 'lc_grass', 'lc_crop', 'lc_bare',
            'pct_within_1km_river', 'pct_water', 'pct_occasional_water', 'pop_density',
            'county_flood_rate']

In [9]:
#number of selected features
len(features)

24

### 2.2 Keeping the 30 Columns

The cells below keep only the named columns. This is a check: if the file already matches, nothing changes.

In [10]:
#keeping the keys, the target, flood_this_month and the 24 features
keep_columns = keys + [target, baseline] + features
df = df[keep_columns]

In [11]:
#shape after keeping the columns
df.shape

(12690, 30)

#### Observation

Keeping the 30 named columns leaves the shape at (12690, 30), the same as the file we loaded, so the file already has exactly these columns.

### 2.3 Checking for Leakage

The target and the four key columns must never be features. `flood_this_month` is allowed as a feature, because it is known at the end of the month we predict from (Section 4.3 checks this). The next cell lists any feature that is also a key or the target, so the list must be empty.

In [12]:
#checking that no key or the target is in the features
print(set(features) & set(keys + [target]))

set()


#### Observation

The leakage check prints an empty set, so none of the features is a key or the target. `len(features)` prints 24, and the table has 30 columns: 4 keys, the target, `flood_this_month` and the 24 features. `flood_this_month` is kept as a column and is used in the model notebooks both as a feature and as the persistence baseline.

## 3. Selecting the Rows

### 3.1 Checking for Missing Values

The file should already start in July 2002 and end in December 2024. The next cells show where the missing values are, so the choice of the time period can be checked against the data.

In [13]:
#columns with missing values in the full table
missing = df.isna().sum()
missing[missing > 0]

Series([], dtype: int64)

In [14]:
#years of the rows with no NDVI
df[df['ndvi_pct_normal'].isna()]['year'].value_counts().sort_index()

Series([], Name: count, dtype: int64)

In [15]:
#year and month of the rows with no next-month label
df[df['flood_next_month'].isna()][['year', 'month']].drop_duplicates()

,year,month


#### Observation

No column has missing values (the first output is empty), no row has a missing NDVI value and no row has a missing next-month label. This is what we expect from the reduced file: the months before NDVI starts and the months at the end that have no next-month label are no longer in it. Section 3.2 checks the first and the last month.

### 3.2 Keeping July 2002 to December 2024

NDVI starts in July 2002 and the labels of 2025 are incomplete, so the table should only hold July 2002 to December 2024. The cells below apply this filter as a check: if the file already matches, no row is dropped.

In [16]:
#start of the period: July 2002 or later
after_start = (df['year'] > 2002) | ((df['year'] == 2002) & (df['month'] >= 7))

In [17]:
#end of the period: December 2024 or earlier
before_end = df['year'] <= 2024

In [18]:
#keeping only the rows inside the period
df = df[after_start & before_end]
df = df.reset_index(drop=True)

In [19]:
#shape after filtering the rows
df.shape

(12690, 30)

In [20]:
#checking the first and the last month kept
print(df['year'].min(), df[df['year'] == df['year'].min()]['month'].min())
print(df['year'].max(), df[df['year'] == df['year'].max()]['month'].max())

2002 7
2024 12


In [21]:
#missing values left after filtering
print(df.isna().sum().sum())

0


In [22]:
#the target is a whole number now that the missing labels are gone
df['flood_next_month'] = df['flood_next_month'].astype(int)

#### Observation

After keeping July 2002 to December 2024 the table still has 12,690 rows and 30 columns, so no row was dropped and the file already matches the period. The first month is month 7 of 2002 and the last is month 12 of 2024, and no missing values are left.

## 4. Train/Test Split

### 4.1 Splitting by Time

The split is by time, not random. The aim is to forecast next month from this month, so the model must be tested on months that come after the months it learned from. A random split would mix months from the same years into both sets, which lets the model learn from months right next to the ones it is tested on and makes the results look better than they would be in real use.

- Train: July 2002 to December 2019
- Test: January 2020 to December 2024

In [23]:
#train = July 2002 to December 2019
train = df[df['year'] <= 2019].copy()

In [24]:
#test = January 2020 to December 2024
test = df[df['year'] >= 2020].copy()

### 4.2 Checking the Split

In [25]:
#shapes of the two sets
print(train.shape)
print(test.shape)

(9870, 30)
(2820, 30)


In [26]:
#number of months with no flood (0) and with a flood (1) next month
print(train['flood_next_month'].value_counts())
print(test['flood_next_month'].value_counts())

flood_next_month
0    8988
1     882
Name: count, dtype: int64
flood_next_month
0    2676
1     144
Name: count, dtype: int64


In [27]:
#share of flood rows in each set, in percent
print(round(train['flood_next_month'].mean() * 100, 1))
print(round(test['flood_next_month'].mean() * 100, 1))

8.9
5.1


In [28]:
#last year of the training set and first year of the test set
print(train['year'].max(), test['year'].min())

2019 2020


In [29]:
#number of counties in each set
print(train['pcode'].nunique())
print(test['pcode'].nunique())

47
47


In [30]:
#number of months for each county in each set
print(train.groupby('pcode').size().unique())
print(test.groupby('pcode').size().unique())

[210]
[60]


In [31]:
#number of floods (flood next month) in each year, in train and in test
print(train.groupby('year')['flood_next_month'].sum())
print(test.groupby('year')['flood_next_month'].sum())

year
2002     14
2003     48
2004     29
2005     14
2006     75
2007    104
2008     47
2009     73
2010     78
2011     73
2012     87
2013     54
2014     17
2015    102
2016     26
2017      6
2018     26
2019      9
Name: flood_next_month, dtype: int64
year
2020    39
2021    30
2022     3
2023    31
2024    41
Name: flood_next_month, dtype: int64


#### Observation

The training set has 9,870 rows (210 months for each of the 47 counties) and the test set has 2,820 rows (60 months for each county). The training set has 882 flood rows (8.9%) and 8,988 non-flood rows, and the test set has 144 flood rows (5.1%) and 2,676 non-flood rows. The last training year is 2019 and the first test year is 2020, so every test month comes after every training month, and both sets contain all 47 counties. Floods are rarer in the test period than in the training period, so the test set has only 144 floods to find.

The floods are not spread evenly over the years. In training, 2007 has the most floods (104), followed by 2015 (102). In the test set, 2022 has only 3 floods, while the other four years have between 30 and 41.

### 4.3 Checking flood_this_month as a Feature

We predict at the end of month t whether a county floods in month t+1. `flood_this_month` says whether it flooded in month t, which is already known when we predict, so it is a lag of the target, like `rain_lag1`. The next two cells check this: for each county, `flood_next_month` of the previous month should be exactly `flood_this_month` of the current month.

In [32]:
#checking flood_this_month is the previous month's flood_next_month (train)
check = train.sort_values(['pcode', 'year', 'month'])
check['prev_next'] = check.groupby('pcode')['flood_next_month'].shift(1)
check = check.dropna(subset=['prev_next'])
print(len(check))
print((check['flood_this_month'] == check['prev_next']).mean())

9823
1.0


In [33]:
#checking flood_this_month is the previous month's flood_next_month (test)
check = test.sort_values(['pcode', 'year', 'month'])
check['prev_next'] = check.groupby('pcode')['flood_next_month'].shift(1)
check = check.dropna(subset=['prev_next'])
print(len(check))
print((check['flood_this_month'] == check['prev_next']).mean())

2773
1.0


In [34]:
#next-month flood rate by whether the county flooded this month, and new vs continuing floods
print(train.groupby('flood_this_month')['flood_next_month'].agg(['count', 'sum', 'mean']))
print(train[train['flood_next_month'] == 1]['flood_this_month'].value_counts())

                  count  sum      mean
flood_this_month                      
0                  8988  474  0.052737
1                   882  408  0.462585
flood_this_month
0    474
1    408
Name: count, dtype: int64


#### Observation

The check keeps 9,823 training rows and 2,773 test rows (47 fewer than each set, because the first month of each county has no earlier row), and in both sets the share of rows where `flood_this_month` equals the shifted `flood_next_month` is 1.0. So `flood_this_month` is exactly the target of the previous month: a clean lag that is known at the end of the month we predict from, like `rain_lag1`.

In the training rows, a county that flooded this month (882 rows) floods again next month in 408 of them (mean 0.463), while a county that did not flood this month (8,988 rows) floods next month in 474 (mean 0.053). So a flood this month makes a flood next month much more likely. Of the 882 next-month floods in training, 408 continue a flood that is already happening (`flood_this_month` is 1) and 474 are new floods (`flood_this_month` is 0).

Two limits of using `flood_this_month` as a feature: (1) it assumes this month's floods are known by the end of the month, which needs fast reporting, and (2) because many floods last more than one month, close to half of the next-month floods are continuations, so the models may partly "predict" floods that are already happening. The model notebooks therefore score new floods and continuing floods separately on the test set.

## 5. Fixing the Leak in county_flood_rate

### 5.1 Why It Leaks

The `county_flood_rate` that comes with the table was made from the 2000 to 2019 labels. The training rows are inside that period, so the feature has already seen the training answers. We replace it with the flood rate of each county calculated from the training rows only: the mean of `flood_this_month` for each `pcode` in `train`.

In [35]:
#the shipped county_flood_rate of each county
old_rate = train.groupby('pcode')['county_flood_rate'].first()

In [36]:
#checking the shipped rate is the same on every row of a county (should print 1)
print(train.groupby('pcode')['county_flood_rate'].nunique().max())

1


### 5.2 Recomputing From the Training Rows Only

In [37]:
#flood rate of each county from the training rows only
new_rate = train.groupby('pcode')['flood_this_month'].mean()

In [38]:
#summary of the new rates (47 counties)
new_rate.describe()

count    47.000000
mean      0.089362
std       0.052861
min       0.004762
25%       0.052381
50%       0.090476
75%       0.123810
max       0.223810
Name: flood_this_month, dtype: float64

In [39]:
#how far the shipped rate is from the new rate, and how strongly they are related
print(round((old_rate - new_rate).abs().mean(), 4))
print(round((old_rate - new_rate).abs().max(), 4))
print(round(old_rate.corr(new_rate), 3))

0.0104
0.0308
0.997


### 5.3 Mapping the Rate Onto Train and Test

The same value of each county is put on both sets. The test set does not get its own rate, because that would use the test answers.

In [40]:
#replacing county_flood_rate in train and test with the training-only rate
train['county_flood_rate'] = train['pcode'].map(new_rate)
test['county_flood_rate'] = test['pcode'].map(new_rate)

In [41]:
#checking that no county is missing a rate
print(train['county_flood_rate'].isna().sum())
print(test['county_flood_rate'].isna().sum())

0
0


In [42]:
#checking that every county has the same rate in train and test
print((train.groupby('pcode')['county_flood_rate'].first() == test.groupby('pcode')['county_flood_rate'].first()).all())

True


#### Observation

The shipped `county_flood_rate` has one value for each county (the check prints 1). The new training-only rates range from 0.005 to 0.224 across the 47 counties, with a mean of 0.089, which is close to the 8.9% flood share of the training set. The shipped rate is on average 0.010 away from the new rate (at most 0.031) and the two are strongly related (correlation 0.997), so the fix changes the values only a little, but the new rate is the only one that has not seen the test years. No county is missing a rate in train or in test (0 and 0), and the last check prints True, so each county has the same rate in both sets. One small limit remains: each training row's own outcome is part of the months used to calculate the rate of its county, so it has a very small part in that rate. The test rows are not touched by this. The rate is kept as it is, and the model notebooks list this as a limit.

## 6. Missing Values and Scaling

Nothing is imputed or scaled in this notebook. Scaling goes inside the pipeline of each model notebook, so it is learned from the training rows only and the test set never influences it. The next cell only checks that there are no missing values to deal with.

In [43]:
#missing values in train and in test
print(train.isna().sum().sum())
print(test.isna().sum().sum())

0
0


#### Observation

Both sets have 0 missing values, so nothing needs to be imputed here, and the features are left unscaled for the pipeline in the model notebooks.

## 7. Saving the Split

The training and test sets are saved with all 30 columns, so every model notebook loads the same files. The models use the 24 features and `flood_this_month` as features and `flood_next_month` as the target, and `flood_this_month` is also the persistence baseline.

In [44]:
#creating the data folder if it does not exist yet
os.makedirs('../data', exist_ok=True)

In [45]:
#saving the training and test sets
train.to_csv('../data/train.csv', index=False)
test.to_csv('../data/test.csv', index=False)

In [46]:
#checking the saved files (pcode as text)
train_check = pd.read_csv('../data/train.csv', dtype={'pcode': str})
test_check = pd.read_csv('../data/test.csv', dtype={'pcode': str})
print(train_check.shape)
print(test_check.shape)

(9870, 30)
(2820, 30)


In [47]:
#checking the two files have the same columns and no missing values
print(list(train_check.columns) == list(test_check.columns))
print(train_check.isna().sum().sum() + test_check.isna().sum().sum())

True
0


In [48]:
#columns of the saved training file
print(train_check.columns.tolist())

['pcode', 'county', 'year', 'month', 'flood_next_month', 'flood_this_month', 'rain_mm', 'rain_lag1', 'rain_lag2', 'rain_pct_normal', 'neighbour_rain', 'ndvi_pct_normal', 'oni', 'dmi', 'month_sin', 'month_cos', 'soil_moisture_l1', 'runoff_rank', 'discharge_rank', 'pct_flat', 'area_km2', 'lc_tree', 'lc_grass', 'lc_crop', 'lc_bare', 'pct_within_1km_river', 'pct_water', 'pct_occasional_water', 'pop_density', 'county_flood_rate']


In [49]:
#flood rows in the saved files
print(train_check['flood_next_month'].sum())
print(test_check['flood_next_month'].sum())

882
144


#### Observation

`train.csv` has 9,870 rows and 30 columns and `test.csv` has 2,820 rows and 30 columns. Both files have the same columns in the same order (True) and no missing values (0), and they hold 882 and 144 flood rows, so they match the split in Section 4. The 30 columns are the 4 keys, `flood_next_month`, `flood_this_month` and the 24 features, ending with `county_flood_rate`. The next notebook, `LogisticRegression.ipynb`, trains the logistic regression model on these files.

## 8. Summary of Preprocessing

- **File read:** `model_table.csv` has 12,690 rows and 30 columns and no missing values. It was reduced on purpose from the original table (14,664 rows and 70 columns) to these 30 columns and to July 2002 to December 2024, so it is easier to open and navigate. This notebook only reads it and never writes to it.
- **Columns:** the 30 columns are 4 keys, the target `flood_next_month`, `flood_this_month` and 24 features. The checks in Section 2 show that the file already has exactly these columns. `flood_this_month` is both a model feature and the persistence baseline, because it is already known at the end of the month we predict from (Section 4.3). Only the keys and the target are kept out of the features.
- **Rows:** July 2002 to December 2024 (12,690 rows, 47 counties). The checks in Section 3 show that no row was dropped and that no value is missing.
- **Split by time:** train July 2002 to December 2019 (9,870 rows, 882 floods, 8.9%) and test January 2020 to December 2024 (2,820 rows, 144 floods, 5.1%). In the test set 2022 has only 3 floods.
- **Leak fixed:** `county_flood_rate` is the flood rate of each county from the training rows only, with the same value on train and test. It is kept as it is. It uses all the training rows, so it is slightly optimistic inside cross-validation.
- **Not done here:** no scaling and no imputing (there are no missing values). Scaling goes inside the model pipelines.
- **Saved:** `data/train.csv` and `data/test.csv`. `model_table.csv` is only read and is not changed. None of the files is committed, because of the EM-DAT licence.
- **Feature selection:** the 24 features were chosen in the EDA notebooks using only the training rows (July 2002 to December 2019), so the test years were not used to choose them.

The next notebook, `LogisticRegression.ipynb`, trains the logistic regression model on `train.csv` and compares it with the persistence baseline on `test.csv`.